In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from openai import APIConnectionError, APIStatusError, APITimeoutError, OpenAI, RateLimitError
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()


# ============================================================================
# 05. LLM EVALUATION ON A REPRODUCIBLE STRATIFIED SUBSET
# ============================================================================

BASE_DIR = Path("..").resolve()
INPUT_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
RESULTS_DIR = BASE_DIR / "data/processed/results"

EVAL_SUBSET_PATH = RESULTS_DIR / "llm_eval_subset.csv"
OUTPUT_RESULTS_PATH = RESULTS_DIR / "llm_predictions.csv"
PROGRESS_PATH = RESULTS_DIR / "llm_predictions_progress.csv"

VALID_L1S = ["DEU", "FRA", "ITA", "JPN", "KOR", "SPA", "TUR", "ZHO"]
SAMPLE_SIZE_PER_LABEL_DOMAIN = 50
RANDOM_STATE = 42

MODEL_NAME = "gpt-4o-mini"
MAX_COMPLETION_TOKENS = 80
DRY_RUN_N = 16
RUN_FULL_EVALUATION = True
MAX_RETRIES = 5

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment.")

client = OpenAI(api_key=api_key)

SYSTEM_PROMPT = """You are an expert computational linguist evaluating Native Language Identification from content-word-masked English essays.

The essay uses these placeholders:
- N = noun
- P = proper noun
- V = verb
- A = adjective
- R = adverb

Function words, punctuation, pronouns, determiners, auxiliaries, conjunctions, prepositions, and structural markers such as <P> and <S> remain visible.

Choose exactly one native-language code from:
DEU, FRA, ITA, JPN, KOR, SPA, TUR, ZHO.

Return only the requested JSON object."""

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "nli_prediction",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "predicted_l1": {"type": "string", "enum": VALID_L1S},
                "confidence_score": {"type": "number", "minimum": 0, "maximum": 1},
            },
            "required": ["predicted_l1", "confidence_score"],
            "additionalProperties": False,
        },
    },
}


def load_or_create_eval_subset() -> pd.DataFrame:
    if EVAL_SUBSET_PATH.exists():
        print(f"Loading existing evaluation subset: {EVAL_SUBSET_PATH}")
        return pd.read_csv(EVAL_SUBSET_PATH)

    df = pd.read_csv(INPUT_CSV_PATH)
    df["label"] = df["label"].astype(str).str.strip().str.upper()
    df["dataset"] = df["dataset"].astype(str).str.strip().str.upper()
    df["split"] = df["split"].astype(str).str.strip().str.lower()
    df["masked_text"] = df["masked_text"].fillna("").astype(str)

    filtered = df[df["label"].isin(VALID_L1S)].copy()
    toefl_test = filtered[(filtered["dataset"] == "TOEFL11") & (filtered["split"] == "test")]
    icle_test = filtered[filtered["dataset"] == "ICLEV3"]

    sampled_parts = []
    for domain_name, domain_df in [("toefl11_test", toefl_test), ("iclev3_external_test", icle_test)]:
        counts = domain_df.groupby("label").size()
        too_small = counts[counts < SAMPLE_SIZE_PER_LABEL_DOMAIN]
        if not too_small.empty:
            raise ValueError(
                f"{domain_name} has labels with fewer than {SAMPLE_SIZE_PER_LABEL_DOMAIN} rows: "
                f"{too_small.to_dict()}"
            )

        part = (
            domain_df.groupby("label", group_keys=False)
            .sample(n=SAMPLE_SIZE_PER_LABEL_DOMAIN, random_state=RANDOM_STATE)
            .copy()
        )
        part["eval_partition"] = domain_name
        sampled_parts.append(part)

    eval_df = pd.concat(sampled_parts, ignore_index=True)
    eval_df = eval_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    eval_df.insert(0, "eval_id", [f"EVAL_{i:04d}" for i in range(len(eval_df))])

    keep_columns = ["eval_id", "doc_id", "dataset", "split", "eval_partition", "label", "masked_text"]
    eval_df[keep_columns].to_csv(EVAL_SUBSET_PATH, index=False)
    print(f"Saved evaluation subset: {EVAL_SUBSET_PATH}")
    return eval_df[keep_columns]


def call_llm(masked_text: str) -> dict:
    user_prompt = f"Masked essay:\n\n{masked_text}"

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=RESPONSE_FORMAT,
                temperature=0,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )

            raw_content = response.choices[0].message.content
            parsed = json.loads(raw_content)

            usage = response.usage
            return {
                "predicted_l1": parsed["predicted_l1"],
                "confidence_score": parsed["confidence_score"],
                "raw_response": raw_content,
                "prompt_tokens": usage.prompt_tokens if usage else None,
                "completion_tokens": usage.completion_tokens if usage else None,
                "total_tokens": usage.total_tokens if usage else None,
                "error": "",
            }

        except (RateLimitError, APITimeoutError, APIConnectionError, APIStatusError) as e:
            wait_seconds = min(60, 2**attempt)
            if attempt == MAX_RETRIES - 1:
                return {
                    "predicted_l1": None,
                    "confidence_score": None,
                    "raw_response": "",
                    "prompt_tokens": None,
                    "completion_tokens": None,
                    "total_tokens": None,
                    "error": repr(e),
                }
            time.sleep(wait_seconds)


eval_df = load_or_create_eval_subset()
print(eval_df.groupby(["eval_partition", "label"]).size().unstack(fill_value=0))

existing = pd.DataFrame()
completed_ids = set()
if PROGRESS_PATH.exists():
    existing = pd.read_csv(PROGRESS_PATH)
    completed_ids = set(existing["eval_id"])
    print(f"Resuming from progress file. Completed rows: {len(completed_ids)}")

rows = [] if existing.empty else existing.to_dict("records")
new_calls_made = 0

dry_run_rows = eval_df[~eval_df["eval_id"].isin(completed_ids)].head(DRY_RUN_N)
if not dry_run_rows.empty and len(completed_ids) == 0:
    print(f"Running {len(dry_run_rows)} dry-run calls before full evaluation...")
    print("RUN_FULL_EVALUATION is False, so this notebook will stop after the dry run.")

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    if row["eval_id"] in completed_ids:
        continue

    result = call_llm(row["masked_text"])
    output_row = {
        "eval_id": row["eval_id"],
        "doc_id": row["doc_id"],
        "dataset": row["dataset"],
        "split": row["split"],
        "eval_partition": row["eval_partition"],
        "true_l1": row["label"],
        "model": MODEL_NAME,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        **result,
    }
    rows.append(output_row)
    new_calls_made += 1

    pd.DataFrame(rows).to_csv(PROGRESS_PATH, index=False)

    if new_calls_made == DRY_RUN_N:
        progress_df = pd.DataFrame(rows)
        token_total = progress_df["total_tokens"].dropna().sum()
        mean_tokens = token_total / len(progress_df)
        estimated_total_tokens = mean_tokens * len(eval_df)
        print(f"Dry run complete. Mean tokens/call: {mean_tokens:.1f}")
        print(f"Estimated total tokens for {len(eval_df)} calls: {estimated_total_tokens:.0f}")
        print("Inspect cost manually before continuing.")
        print("To run all remaining rows, set RUN_FULL_EVALUATION = True and rerun this notebook.")
        if not RUN_FULL_EVALUATION:
            break

results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_RESULTS_PATH, index=False)

print(f"Saved LLM predictions: {OUTPUT_RESULTS_PATH}")
print(results_df["error"].replace("", pd.NA).dropna().head())